# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [65]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Aug 20 05:49:11 PM 2026"

In [66]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,778659,41.6,1473301,78.7,1473301,78.7
Vcells,1539347,11.8,124166515,947.4,252391628,1925.6


In [67]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Aqui debe cargar SU semilla primigenia

In [68]:
PARAM <- list()
PARAM$semilla_primigenia <- 123457

# parametros  arbol
# entreno cada arbol con solo 50% de las variables variables
#  por ahora, es fijo
PARAM$feature_fraction <- 0.25

PARAM$rpart$cp <- -1
PARAM$rpart$minsplit <- 700
PARAM$rpart$minbucket <- 2
PARAM$rpart$maxdepth <- 16


# voy a generar 32 arboles,
#  a mas arboles mas tiempo de proceso y MEJOR MODELO,
#  pero ganancias marginales
PARAM$num_trees_max <- 32

In [69]:
# Valores que vamos a probar
grid_minbucket <- c(2, 4, 6)
grid_minsplit <- c(500, 700)
grid_maxdepth <- c(12, 16)
grid_feature_fraction <- c(0.25, 0.5)

grid <- expand.grid(
  minbucket = grid_minbucket,
  minsplit = grid_minsplit,
  maxdepth = grid_maxdepth,
  feature_fraction = grid_feature_fraction
)

message("Cantidad de configuraciones: ", nrow(grid))

Cantidad de configuraciones: 24



In [70]:
# ============================================================
# CARPETAS PARA CADA CONFIGURACION
# ============================================================

base_grid_dir <- "/content/buckets/b1/exp/exp4021/gridsearch"

dir.create(
  base_grid_dir,
  showWarnings = FALSE,
  recursive = TRUE
)

for (configuracion in 1:nrow(grid)) {

  carpeta_config <- file.path(
    base_grid_dir,
    paste0("config_", sprintf("%02d", configuracion))
  )

  dir.create(
    carpeta_config,
    showWarnings = FALSE,
    recursive = TRUE
  )
}

message("Carpetas creadas: ", nrow(grid))

Carpetas creadas: 24



In [71]:
# ============================================================
# VERIFICAR LAS CONFIGURACIONES
# ============================================================

for (configuracion in 1:nrow(grid)) {

  message(
    "Configuración ", configuracion,
    " | minbucket = ", grid$minbucket[configuracion],
    " | minsplit = ", grid$minsplit[configuracion],
    " | maxdepth = ", grid$maxdepth[configuracion],
    " | feature_fraction = ", grid$feature_fraction[configuracion]
  )
}

Configuración 1 | minbucket = 2 | minsplit = 500 | maxdepth = 12 | feature_fraction = 0.25

Configuración 2 | minbucket = 4 | minsplit = 500 | maxdepth = 12 | feature_fraction = 0.25

Configuración 3 | minbucket = 6 | minsplit = 500 | maxdepth = 12 | feature_fraction = 0.25

Configuración 4 | minbucket = 2 | minsplit = 700 | maxdepth = 12 | feature_fraction = 0.25

Configuración 5 | minbucket = 4 | minsplit = 700 | maxdepth = 12 | feature_fraction = 0.25

Configuración 6 | minbucket = 6 | minsplit = 700 | maxdepth = 12 | feature_fraction = 0.25

Configuración 7 | minbucket = 2 | minsplit = 500 | maxdepth = 16 | feature_fraction = 0.25

Configuración 8 | minbucket = 4 | minsplit = 500 | maxdepth = 16 | feature_fraction = 0.25

Configuración 9 | minbucket = 6 | minsplit = 500 | maxdepth = 16 | feature_fraction = 0.25

Configuración 10 | minbucket = 2 | minsplit = 700 | maxdepth = 16 | feature_fraction = 0.25

Configuración 11 | minbucket = 4 | minsplit = 700 | maxdepth = 16 | feature_fra

In [72]:
setwd("/content/buckets/b1/exp")

experimento <- "exp4021"

dir.create(
  experimento,
  showWarnings = FALSE
)

# Entro a la carpeta del experimento
setwd(
  paste0(
    "/content/buckets/b1/exp/",
    experimento
  )
)

message(
  "Carpeta de trabajo: ",
  getwd()
)

Carpeta de trabajo: /content/.drive/My Drive/dmeyf/exp/exp4021



In [73]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [74]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [75]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [76]:
# que tamanos de ensemble grabo a disco
grabar <- c(32)

In [77]:
set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

In [78]:
# ============================================================
# CHECKPOINTS
# ============================================================

checkpoint_dir <- base_grid_dir

# Creo una carpeta independiente para cada configuración
for (configuracion in 1:1) {

  carpeta_config <- file.path(
    checkpoint_dir,
    paste0(
      "config_",
      sprintf("%02d", configuracion)
    )
  )

  dir.create(
    carpeta_config,
    showWarnings = FALSE,
    recursive = TRUE
  )
}

list.dirs(
  checkpoint_dir,
  recursive = FALSE
)









[1] "/content/buckets/b1/exp/exp4021/gridsearch/config_01"
 [2] "/content/buckets/b1/exp/exp4021/gridsearch/config_02"
 [3] "/content/buckets/b1/exp/exp4021/gridsearch/config_03"
 [4] "/content/buckets/b1/exp/exp4021/gridsearch/config_04"
 [5] "/content/buckets/b1/exp/exp4021/gridsearch/config_05"
 [6] "/content/buckets/b1/exp/exp4021/gridsearch/config_06"
 [7] "/content/buckets/b1/exp/exp4021/gridsearch/config_07"
 [8] "/content/buckets/b1/exp/exp4021/gridsearch/config_08"
 [9] "/content/buckets/b1/exp/exp4021/gridsearch/config_09"
[10] "/content/buckets/b1/exp/exp4021/gridsearch/config_10"
[11] "/content/buckets/b1/exp/exp4021/gridsearch/config_11"
[12] "/content/buckets/b1/exp/exp4021/gridsearch/config_12"
[13] "/content/buckets/b1/exp/exp4021/gridsearch/config_13"
[14] "/content/buckets/b1/exp/exp4021/gridsearch/config_14"
[15] "/content/buckets/b1/exp/exp4021/gridsearch/config_15"
[16] "/content/buckets/b1/exp/exp4021/gridsearch/config_16"
[17] "/content/buckets/b1/exp/exp4021/gridsearch/config_17"
[18] "/content/buckets/b1/exp/exp4021/gridsearch/config_18"
[19] "/content/buckets/b1/exp/exp4021/gridsearch/config_19"
[20] "/content/buckets/b1/exp/exp4021/gridsearch/config_20"
[21] "/content/buckets/b1/exp/exp4021/gridsearch/config_21"
[22] "/content/buckets/b1/exp/exp4021/gridsearch/config_22"
[23] "/content/buckets/b1/exp/exp4021/gridsearch/config_23"
[24] "/content/buckets/b1/exp/exp4021/gridsearch/config_24"

In [ ]:
# ============================================================
# GRID SEARCH - 24 CONFIGURACIONES CON CHECKPOINT
# ============================================================

for (configuracion in 1:nrow(grid)) {

  # ----------------------------------------------------------
  # PARAMETROS DE ESTA CONFIGURACION
  # ----------------------------------------------------------

  PARAM$rpart$minbucket <- grid$minbucket[configuracion]
  PARAM$rpart$minsplit <- grid$minsplit[configuracion]
  PARAM$rpart$maxdepth <- grid$maxdepth[configuracion]
  PARAM$feature_fraction <- grid$feature_fraction[configuracion]

  message("")
  message("==============================================")
  message(
    "CONFIGURACION ",
    configuracion,
    " DE ",
    nrow(grid)
  )
  message(
    "minbucket = ", PARAM$rpart$minbucket,
    " | minsplit = ", PARAM$rpart$minsplit,
    " | maxdepth = ", PARAM$rpart$maxdepth,
    " | feature_fraction = ", PARAM$feature_fraction
  )
  message("==============================================")


  # ----------------------------------------------------------
  # CARPETA DE ESTA CONFIGURACION
  # ----------------------------------------------------------

  carpeta_config <- file.path(
    base_grid_dir,
    paste0(
      "config_",
      sprintf("%02d", configuracion)
    )
  )

  dir.create(
    carpeta_config,
    showWarnings = FALSE,
    recursive = TRUE
  )


  # ----------------------------------------------------------
  # BUSCO EL ULTIMO CHECKPOINT
  # ----------------------------------------------------------

  archivos_checkpoint <- list.files(
    carpeta_config,
    pattern = "^checkpoint_[0-9]+\\.rds$",
    full.names = TRUE
  )

  if (length(archivos_checkpoint) > 0) {

    numeros_checkpoint <- as.integer(
      sub(
        "checkpoint_([0-9]+)\\.rds",
        "\\1",
        basename(archivos_checkpoint)
      )
    )

    ultimo_arbol <- max(numeros_checkpoint)

    archivo_checkpoint <- file.path(
      carpeta_config,
      paste0(
        "checkpoint_",
        ultimo_arbol,
        ".rds"
      )
    )

    estado <- readRDS(
      archivo_checkpoint
    )

    tb_prediccion <- estado$tb_prediccion

    arbol_inicio <- estado$arbol_siguiente

    # Recupero exactamente el estado de la semilla
    .Random.seed <- estado$random_seed

    message(
      "Checkpoint encontrado: árbol ",
      ultimo_arbol
    )

    message(
      "Continuando desde árbol ",
      arbol_inicio
    )

  } else {

    message(
      "No hay checkpoint. Comenzando desde el árbol 1."
    )

    tb_prediccion <- dfuture[
      ,
      list(numero_de_cliente)
    ]

    tb_prediccion[
      ,
      prob_acumulada := 0
    ]

    # Semilla inicial de esta configuracion
    set.seed(
      PARAM$semilla_primigenia
    )

    arbol_inicio <- 1
  }


  # ----------------------------------------------------------
  # SI YA TERMINO ESTA CONFIGURACION
  # ----------------------------------------------------------

  if (arbol_inicio > PARAM$num_trees_max) {

    message(
      "La configuracion ",
      configuracion,
      " ya fue completada."
    )

    next
  }


  # ----------------------------------------------------------
  # GENERACION DE LOS ARBOLES
  # ----------------------------------------------------------

  for (
    arbolito in arbol_inicio:PARAM$num_trees_max
  ) {

    message(
      "Configuracion ",
      configuracion,
      " - Arbol ",
      arbolito
    )


    # --------------------------------------------------------
    # CANTIDAD DE CAMPOS A UTILIZAR
    # --------------------------------------------------------

    qty_campos_a_utilizar <- as.integer(
      length(campos_buenos) *
        PARAM$feature_fraction
    )


    # --------------------------------------------------------
    # SELECCIONO CAMPOS AL AZAR
    # --------------------------------------------------------

    campos_random <- sample(
      campos_buenos,
      qty_campos_a_utilizar
    )

    campos_random <- paste(
      campos_random,
      collapse = " + "
    )


    # --------------------------------------------------------
    # FORMULA
    # --------------------------------------------------------

    formulita <- paste0(
      "clase_ternaria ~ ",
      campos_random
    )


    # --------------------------------------------------------
    # GENERO EL ARBOL
    # --------------------------------------------------------

    modelo <- rpart(
      formulita,
      data = dtrain,
      xval = 0,
      control = PARAM$rpart
    )


    # --------------------------------------------------------
    # PREDICCION
    # --------------------------------------------------------

    prediccion <- predict(
      modelo,
      dfuture,
      type = "prob"
    )


    # --------------------------------------------------------
    # ACUMULO PROBABILIDAD
    # --------------------------------------------------------

    tb_prediccion[
      ,
      prob_acumulada :=
        prob_acumulada +
        prediccion[, "BAJA+2"]
    ]


    # ========================================================
    # CHECKPOINT CADA 5 ARBOLES
    # ========================================================

    if (
      arbolito %% 5 == 0 ||
      arbolito == PARAM$num_trees_max
    ) {

      archivo_checkpoint <- file.path(
        carpeta_config,
        paste0(
          "checkpoint_",
          arbolito,
          ".rds"
        )
      )

      estado <- list(
        tb_prediccion = tb_prediccion,
        arbol_siguiente = arbolito + 1,
        random_seed = .Random.seed
      )

      saveRDS(
        estado,
        archivo_checkpoint
      )

      message(
        "Checkpoint guardado: árbol ",
        arbolito
      )
    }


    # ========================================================
    # GRABAR RESULTADO FINAL - ARBOL 32
    # ========================================================

    if (arbolito %in% grabar) {

      umbral_corte <- (
        1 / 40
      ) * arbolito

      tb_prediccion[
        ,
        Predicted :=
          as.numeric(
            prob_acumulada >
              umbral_corte
          )
      ]


      # ------------------------------------------------------
      # NOMBRE DEL ARCHIVO
      # ------------------------------------------------------

      archivo_kaggle <- file.path(
        carpeta_config,
        paste0(
          "KA421_",
          "mb", PARAM$rpart$minbucket,
          "_ms", PARAM$rpart$minsplit,
          "_md", PARAM$rpart$maxdepth,
          "_ff", PARAM$feature_fraction,
          "_",
          sprintf("%.3d", arbolito),
          ".csv"
        )
      )


      # ------------------------------------------------------
      # GRABO EL ARCHIVO
      # ------------------------------------------------------

      fwrite(
        tb_prediccion[
          ,
          list(
            numero_de_cliente,
            Predicted
          )
        ],
        file = archivo_kaggle,
        sep = ","
      )

      message(
        "Archivo generado: ",
        archivo_kaggle
      )


      # ------------------------------------------------------
      # SUBIDA A KAGGLE
      # ------------------------------------------------------

      comando <- "kaggle competitions submit"

      competencia <- "-c utn-2026-inicial"

      arch <- paste(
        "-f",
        shQuote(archivo_kaggle)
      )

      mensaje <- paste0(
        "-m 'config=",
        configuracion,
        " cp=",
        PARAM$rpart$cp,
        " minsplit=",
        PARAM$rpart$minsplit,
        " minbucket=",
        PARAM$rpart$minbucket,
        " maxdepth=",
        PARAM$rpart$maxdepth,
        " feature_fraction=",
        PARAM$feature_fraction,
        "'"
      )

      linea <- paste(
        comando,
        competencia,
        arch,
        mensaje
      )

      salida <- system(
        linea,
        intern = TRUE
      )

      cat(salida)
    }
  }
}




CONFIGURACION 1 DE 24

minbucket = 2 | minsplit = 500 | maxdepth = 12 | feature_fraction = 0.25


Checkpoint encontrado: árbol 10

Continuando desde árbol 11

Configuracion 1 - Arbol 11

Configuracion 1 - Arbol 12



In [ ]:
format(Sys.time(), "%a %b %d %X %Y")



---

